In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatOpenAI()

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza maker go to therapy? Because he had too many topping issues!',
 'explanation': 'This joke is a play on words, as "topping issues" sounds similar to "topping tissues" which could refer to the variety of toppings that a pizza maker has to deal with on a regular basis. Going to therapy implies that the pizza maker is seeking help for his overwhelming amount of toppings, suggesting that he may be struggling to keep up with all the different options and decisions that go into making a pizza. Overall, the joke is meant to be lighthearted and humorous, poking fun at the idea of someone needing therapy for something as seemingly simple as making pizzas.'}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker go to therapy? Because he had too many topping issues!', 'explanation': 'This joke is a play on words, as "topping issues" sounds similar to "topping tissues" which could refer to the variety of toppings that a pizza maker has to deal with on a regular basis. Going to therapy implies that the pizza maker is seeking help for his overwhelming amount of toppings, suggesting that he may be struggling to keep up with all the different options and decisions that go into making a pizza. Overall, the joke is meant to be lighthearted and humorous, poking fun at the idea of someone needing therapy for something as seemingly simple as making pizzas.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12626b-8a60-63ea-8002-c6b4801dc740'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-22T19:38:37.369418+00:00', parent_config={'configurable': {'thread_id'

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker go to therapy? Because he had too many topping issues!', 'explanation': 'This joke is a play on words, as "topping issues" sounds similar to "topping tissues" which could refer to the variety of toppings that a pizza maker has to deal with on a regular basis. Going to therapy implies that the pizza maker is seeking help for his overwhelming amount of toppings, suggesting that he may be struggling to keep up with all the different options and decisions that go into making a pizza. Overall, the joke is meant to be lighthearted and humorous, poking fun at the idea of someone needing therapy for something as seemingly simple as making pizzas.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12626b-8a60-63ea-8002-c6b4801dc740'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-22T19:38:37.369418+00:00', parent_config={'configurable': {'thread_id

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the noodle go to the doctor? Because it was feeling pasta-tively al dente!',
 'explanation': 'This joke is a play on words involving the Italian pasta dish spaghetti. In this case, the noodle, which is a type of pasta, went to the doctor because it was feeling "pasta-tively" al dente. "Pasta-tively" is a play on the word "positively," and "al dente" is an Italian term used to describe pasta that is cooked to be firm to the bite. Thus, the noodle went to the doctor to seek help for its firm and undercooked texture, which is typically not desired in pasta dishes. This joke combines humor with wordplay related to food terms.'}

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker go to therapy? Because he had too many topping issues!', 'explanation': 'This joke is a play on words, as "topping issues" sounds similar to "topping tissues" which could refer to the variety of toppings that a pizza maker has to deal with on a regular basis. Going to therapy implies that the pizza maker is seeking help for his overwhelming amount of toppings, suggesting that he may be struggling to keep up with all the different options and decisions that go into making a pizza. Overall, the joke is meant to be lighthearted and humorous, poking fun at the idea of someone needing therapy for something as seemingly simple as making pizzas.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12626b-8a60-63ea-8002-c6b4801dc740'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-22T19:38:37.369418+00:00', parent_config={'configurable': {'thread_id

Time Travel

In [13]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f12626b-65a7-6dd2-8000-ae38977c6a7d"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f12626b-65a7-6dd2-8000-ae38977c6a7d'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-22T19:38:33.519223+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12626b-659b-6f78-bfff-b2b89a7aaab5'}}, tasks=(PregelTask(id='b0eeed6e-2327-ac28-b89d-45eb673af9bc', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza maker go to therapy? Because he had too many topping issues!'}),), interrupts=())

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f12626b-65a7-6dd2-8000-ae38977c6a7d"}})

Updating State

In [14]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f12626b-65a7-6dd2-8000-ae38977c6a7d", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f126294-bac9-68f0-8001-fedb32c254ca'}}

In [15]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f126294-bac9-68f0-8001-fedb32c254ca'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-03-22T19:57:03.031302+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12626b-65a7-6dd2-8000-ae38977c6a7d'}}, tasks=(PregelTask(id='86b53967-8fa4-dcf3-400f-7accd98f461b', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker go to therapy? Because he had too many topping issues!', 'explanation': 'This joke is a play on words, as "topping issues" sounds similar to "topping tissues" which could refer to the variety of toppings that a pizza maker has to deal with on a regular basis. Going to therapy implies that the pizza maker 

In [16]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f126294-bac9-68f0-8001-fedb32c254ca"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy? \nBecause it had too many layers to peel back!',
 'explanation': 'This joke plays on the idea that samosas are a type of pastry with multiple layers of dough. In therapy, individuals often peel back layers of emotions and thoughts to get to the root of their issues. So, the joke humorously suggests that the samosa needed therapy because it had too many layers to peel back, both literally as a pastry and metaphorically as a complex being.'}